# Глубинное обучение на табличных данных

Начнем конечно с импортов различных, которые понадобятся далее. Также, многое отсюда было реализовано с помощью функций копипаста с семинаров, что мы проходили, помимо этого, семинары дали вдохновление на именно такой пайплайн действий) Стоило упомянуть

In [79]:
import pandas as pd
import numpy as np
import random

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader #чтобы подавать данные батчами

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score #для подсчета качества

import os
import wandb

Объявим класс конфига

In [80]:
class CFG:

# Задаем параметры нашего эксперимента

#   api = "" вписать свой API Wandb но в енв
  project = "Kolesa"# вписать название эксперимента, который предварительно надо создать в Wandb
#   entity = "" ввести свой логин но в енв
  num_epochs = 15 # количество эпох
  train_batch_size = 64 # размер батча обучающей выборки
  test_batch_size = 256 # размер батча тестовой выборки
  lr = 0.001 # learning_rate
  seed = 42 # для функции воспроизводимости
  wandb = False # флаг использования Wandb

Зафиксируем сиды

In [81]:
def seed_everything(seed): # чтобы запуски были стабильными и воспроизводимымми
    random.seed(seed) # фиксируем генератор случайных чисел
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU

In [82]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device # то есть, если есть видеокарта то юзаем cuda, если нет - то юзаем cpu

device(type='cpu')

Загрузим все предобработанные данные 

In [83]:
X_train =pd.read_csv('../data/X_train.csv')
X_test =pd.read_csv('../data/X_test.csv')
y_train =pd.read_csv('../data/y_train.csv')
y_test =pd.read_csv('../data/y_test.csv')

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((4236, 570), (1060, 570), (4236, 1), (1060, 1))

Когда переводил в тензоры выдавало ошибку, потому что нужны флоат, а там видимо от ohe остались bool, переведем это все в флоат и пойдем дальше

In [84]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)
y_train = y_train.astype(float)
y_test = y_test.astype(float)

Переведем все в тензоры

In [85]:
X_train_tensor =torch.FloatTensor(X_train.values)
X_test_tensor =torch.FloatTensor(X_test.values)
y_train_tensor =torch.FloatTensor(y_train.values)
y_test_tensor =torch.FloatTensor(y_test.values)

X_train_tensor.shape, y_train_tensor.shape

(torch.Size([4236, 570]), torch.Size([4236, 1]))

Теперь соединим это обратно в датасеты (train/test), чтобы передавать в модели именно все признаки с таргетом, после чего создадим dataloader чтобы подавать в модель данные батчами/частями

In [86]:
train_ds = TensorDataset(X_train_tensor,y_train_tensor)
test_ds = TensorDataset(X_test_tensor,y_test_tensor)

train_ds

In [87]:
train_loader = DataLoader(train_ds, batch_size= CFG.train_batch_size, shuffle=True) #shuffle тут чтобы между эпохами перемешивались строки и модель не привыкала к порядку
test_loader = DataLoader(test_ds, batch_size= CFG.test_batch_size)


In [88]:
input_size = X_train.shape[1]
input_size 

570

Сейчас построим достаточно базовую модель (вдохновление от 15 семинара). Архитектура простая - 3 полносвязных слоя и все. Далее будем строить несколько более сложных архитектур. Сначала простую чтобы понимать, помогают ли нам новые архитектуры улучшить так скажем 'базовое' качество. 

In [89]:
class Simple(nn.Module): # наследуемся от класса nn.Module
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 128) # первый скрытый слой - 128 нейронов
        self.fc2 = nn.Linear(128, 64) # второй скрытый - 64
        self.fc3 = nn.Linear(64, 1) # третий  выходной- 1 прогноз
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x) 
        return x

Теперь построим более сложную (чуть) архитектуру. Попробуем добавить просто больше скрытых слоев и добавим еще больше нейронов 

Но, при этом, есильно выше риск переобучения, поэтому будем все это сравнивать на качестве теста

In [90]:
class Deep(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 512) #теперь начинаем уменьшение размерности с 512 
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

И, добавим третью модель, тут мы добавим еще дропаут и батчнорм. Первый будет бороться с переобучением за счет отключения определенного количества нейронов во время обучения. Батчнорм будет просто стабилизировать значения внутри срктытых слоев

In [91]:
class Regularized(nn.Module):
    def __init__(self, input_size):
        super().__init__()


        self.net = nn.Sequential(nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.05), #1 скрытый 
                                 nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),nn.Dropout(0.05),  #2 скрытый
                                 nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))    #3 скрытый и 4 выходной
        #sequential тут потому что во первых мы так работали на семинарах, а во-вторых потому что слои и операции в них просто идут по порядку
        #без сложностей , и следующая функция красивая в 2 строки получается )))

    def forward(self, x):
        return self.net(x)

Мы решили добавить еще одну полнсвязную сеть с residual connection. Его преимущество в том, что он сохраняет вход блока и добавляет к выходу. То есь, это хорошо потому что мы по идее не теряем юзфул информацию когда проходим несколько слоев. В нашем случае, это полезно потому что после ohe у нас стало много признаков, и там есть важные признаки ,которые residual mlp поможет учитывать в каких то сложных комбинациях

Сайты, в том числе откуда и вдохновение на код:
- https://discuss.pytorch.org/t/how-to-use-residual-learning-applied-to-fully-connected-networks/98708/5
- https://stackoverflow.com/questions/60817390/implementing-a-simple-resnet-block-with-pytorch

In [92]:
class ResBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        
    def forward(self, x):
        residual = x  # сохраняем вход блока
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        x = x + residual  # прибавляем старый x к новому x
        x = torch.relu(x)
        
        return x

In [93]:
class ResMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 256)
        self.block1 = ResBlock(256)
        self.block2 = ResBlock(256)
        self.fc2 = nn.Linear(256, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.block1(x)
        x = self.block2(x)
        x = self.fc2(x)
        return x

Далее, переходим к следующей части , пока просто зададим функцию потерь и потом напишем фукнцию обучения

In [94]:
criterion = nn.MSELoss()

In [95]:
from tqdm import tqdm

def train(model, device, train_loader, optimizer, criterion, epoch, WANDB): 
    model.train()
    train_loss = 0
    
    for data, target in tqdm(train_loader):
        data = data.to(device)
        target = target.to(device)
        
        optimizer.zero_grad() #обнуяем градиенты
        
        output = model(data) #прямой проход
        loss = criterion(output, target) #считаем ошибку
        
        loss.backward() #обратынй проход
        optimizer.step() #шаг оптимизатором
        train_loss += loss.item()
    
    train_loss = train_loss / len(train_loader)
    
    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(train_loss)) #как в семе, но чуть поправленная, потому что там для картинок

    if WANDB:
        wandb.log({'epoch': epoch,
                   'train_loss': train_loss})
        
    return train_loss

Теперь функция тестирования, оч похожа на сем, но там картинки

In [96]:
def test(model, device, test_loader, criterion, epoch, WANDB):
    model.eval()  
    
    test_loss = 0
    preds = []
    true = []
    
    # показываем, что обучения нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            test_loss += criterion(output, target).item()
            
            preds.append(output.cpu())
            true.append(target.cpu())
    
    test_loss = test_loss / len(test_loader)
    
    preds = torch.cat(preds).numpy().ravel() #тут мы клеим прогнозы в одномерный массив нампай 
    true = torch.cat(true).numpy().ravel()
    
    preds = np.expm1(preds) #возвращаем логарифмированные числа в первоначальное 
    true = np.expm1(true)
    
    mae = mean_absolute_error(true, preds)
    rmse = np.sqrt(mean_squared_error(true, preds))
    r2 = r2_score(true, preds)
    
    tqdm.write('Test set: Average loss: {:.4f}, MAE: {:.2f}, RMSE: {:.2f}, R2: {:.4f}'.format(
       test_loss, mae, rmse, r2))
    
    if WANDB:
        wandb.log({'epoch': epoch,
                   'test_loss': test_loss,
                   'test_mae': mae,
                   'test_rmse': rmse,
                   'test_r2': r2})
    
    return test_loss, mae, rmse, r2

Теперь нужна функция запуска наших экспериментов, и после этого можно будет лицезреть качество выстроенных нами моделей))

In [97]:
# основная функция для экспериментов
def main(model, model_name):

    if CFG.wandb:
        wandb.init(project=CFG.project, entity=CFG.entity, name = model_name, reinit=True) #reinit чтобы каждый экспер записывался как новый
    seed_everything(CFG.seed)  # фиксируем сиды
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr)
    
    train_losses = []
    test_losses = []
    best_mae =10**20
    
    if CFG.wandb:
        wandb.watch(model, log='all') # логируем все (метрики, лоссы, градиенты)


    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train_loss = train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        test_loss, mae, rmse, r2 = test(model, device, test_loader, criterion, epoch, CFG.wandb)
        train_losses.append(train_loss)
        test_losses.append(test_loss)

        if mae < best_mae:
            best_mae = mae
            best_test_loss = test_loss
            best_rmse = rmse
            best_r2 = r2
    
    print('Training is ended!')
    
    
    return model, train_losses, test_losses, best_test_loss, best_mae, best_rmse, best_r2

In [98]:
seed_everything(CFG.seed)
simple_model, simple_train_losses, simple_test_losses, simple_test_loss, simple_mae, simple_rmse, simple_r2 = main(Simple(input_size),'Simple')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 1544.31it/s]



Train set: Average loss: 154.6181
Test set: Average loss: 6.3434, MAE: 175640192.00, RMSE: 442042511.01, R2: -1896.2131

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 1493.67it/s]



Train set: Average loss: 3.1786
Test set: Average loss: 1.8145, MAE: 11987410.00, RMSE: 21867715.81, R2: -3.6430

Epoch: 3


...


Train set: Average loss: 0.0873
Test set: Average loss: 0.2604, MAE: 2957542.75, RMSE: 6070008.05, R2: 0.6423

Epoch: 14


100%|██████████| 67/67 [00:00<00:00, 1510.23it/s]



Train set: Average loss: 0.0755
Test set: Average loss: 0.2579, MAE: 2695520.50, RMSE: 5436531.44, R2: 0.7130

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 1441.76it/s]


Train set: Average loss: 0.0675
Test set: Average loss: 0.2489, MAE: 2573330.25, RMSE: 5338706.86, R2: 0.7233
Training is ended!


In [99]:
seed_everything(CFG.seed)
deep_model,deep_train_losses, deep_test_losses, deep_test_loss, deep_mae,deep_rmse, deep_r2 = main(Deep(input_size), 'Deep')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 704.85it/s]



Train set: Average loss: 69.2995
Test set: Average loss: 2.1083, MAE: 32222486.00, RMSE: 74257529.83, R2: -52.5388

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 711.59it/s]



Train set: Average loss: 0.9805
Test set: Average loss: 0.7312, MAE: 7087168.00, RMSE: 17028565.20, R2: -1.8154

Epoch: 3


...


Train set: Average loss: 0.0504
Test set: Average loss: 0.2077, MAE: 2718278.25, RMSE: 5061229.99, R2: 0.7513

Epoch: 14


100%|██████████| 67/67 [00:00<00:00, 727.82it/s]



Train set: Average loss: 0.0503
Test set: Average loss: 0.2126, MAE: 2460117.50, RMSE: 4967494.40, R2: 0.7604

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 753.20it/s]


Train set: Average loss: 0.0458
Test set: Average loss: 0.2178, MAE: 2360216.25, RMSE: 4495798.50, R2: 0.8038
Training is ended!


In [100]:
seed_everything(CFG.seed)
regularized_model,regularized_train_losses, regularized_test_losses, regularized_test_loss, regularized_mae,regularized_rmse, regularized_r2 = main(
    Regularized(input_size),'Regularized')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 633.03it/s]



Train set: Average loss: 113.1095
Test set: Average loss: 4.6846, MAE: 9226840.00, RMSE: 13238562.21, R2: -0.7016

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 625.55it/s]



Train set: Average loss: 1.7527
Test set: Average loss: 0.4279, MAE: 4988162.50, RMSE: 8243188.55, R2: 0.3403

Epoch: 3


...


Train set: Average loss: 0.5091
Test set: Average loss: 0.2232, MAE: 3962799.25, RMSE: 7092541.87, R2: 0.5116

Epoch: 14


100%|██████████| 67/67 [00:00<00:00, 646.44it/s]



Train set: Average loss: 0.5084
Test set: Average loss: 0.2424, MAE: 4435171.00, RMSE: 8227626.59, R2: 0.3427

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 626.27it/s]


Train set: Average loss: 0.4931
Test set: Average loss: 0.2661, MAE: 4316996.50, RMSE: 7163733.92, R2: 0.5017
Training is ended!


In [ ]:
seed_everything(CFG.seed)
res_model, res_train_losses, res_test_losses, res_test_loss, res_mae, res_rmse, res_r2 = main(ResMLP(input_size), 'ResMLP')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 504.69it/s]



Train set: Average loss: 41.0732
Test set: Average loss: 0.8385, MAE: 9280551.00, RMSE: 26109601.72, R2: -5.6189

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 524.97it/s]



Train set: Average loss: 0.5792
Test set: Average loss: 0.4251, MAE: 5570531.50, RMSE: 13822505.33, R2: -0.8551

Epoch: 3


...


Train set: Average loss: 0.0409
Test set: Average loss: 0.1616, MAE: 3129335.25, RMSE: 5621734.26, R2: 0.6931

Epoch: 14


100%|██████████| 67/67 [00:00<00:00, 521.89it/s]



Train set: Average loss: 0.0402
Test set: Average loss: 0.1549, MAE: 2650481.00, RMSE: 4611449.82, R2: 0.7935

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 524.70it/s]


Train set: Average loss: 0.0431
Test set: Average loss: 0.1522, MAE: 2405865.75, RMSE: 4351271.57, R2: 0.8162
Training is ended!


In [102]:
res = pd.DataFrame([{'model': 'Simple', 'test_loss': simple_test_loss, 'MAE': simple_mae, 'RMSE': simple_rmse, 'R2': simple_r2},
                        {'model': 'Deep', 'test_loss': deep_test_loss, 'MAE': deep_mae, 'RMSE': deep_rmse, 'R2': deep_r2},
                        {'model': 'Regularized', 'test_loss': regularized_test_loss, 'MAE': regularized_mae, 'RMSE': regularized_rmse, 'R2': regularized_r2},
                        {'model': 'ResMLP', 'test_loss': res_test_loss, 'MAE': res_mae, 'RMSE': res_rmse, 'R2': res_r2}])

res

,model,test_loss,MAE,RMSE,R2
0,Simple,0.248894,2573330.25,5.338707e+06,0.723267
1,Deep,0.217791,2360216.25,4.495799e+06,0.803754
2,Regularized,0.208251,3702601.50,5.950960e+06,0.656155
3,ResMLP,0.146880,2333617.75,4.335112e+06,0.817531


In [103]:
res.sort_values('MAE')

,model,test_loss,MAE,RMSE,R2
3,ResMLP,0.146880,2333617.75,4.335112e+06,0.817531
1,Deep,0.217791,2360216.25,4.495799e+06,0.803754
0,Simple,0.248894,2573330.25,5.338707e+06,0.723267
2,Regularized,0.208251,3702601.50,5.950960e+06,0.656155


### Сравнение моделей и выводы

По ходу работы мы обучили 4 модели, от простой базовой полносвязной сети до полносвязной с residual блоками. Основная метрика - считаем MAE, средняя ошибка прогноза цены в тенге. 

По результатам: лучшей стала модель ResMLP с показателями MAE - примерно 2,29 млн тенге, R2 - примерно 0,82, то есть модель лучше остальных отличает цены и в среднем меньше ошибается. Базовая модель(Simple) и модель с бОльшим количеством скрытых слоев(Deep) также дают хороший результат. Худший результат показала модель Regularized, возможно dropout для нашего датасета мешала модели запоминать какие-то важные зависимости в данных или еще что-то, но любые манипуляции с весами не помогали. 